# MILK10k — Step 8: Full pipeline on every training run → mean ± SD across seeds
Finds every attached training run (v1 `stage2`, v2 `stage2v2_s42/43/44`) and re-runs, for each one, exactly the analyses of notebooks 03 → 04b → 05b:
1. **Fusion** under missing inputs (dedicated models + late fusion with modality dropout)
2. **Calibration** (cross-fitted temperature) and **conformal set sizes** per input combination
3. **Dermoscopy-request policy** (rate needed for 95% of the gain; sensitivity at 70% specificity)
4. **Conformal triage** (group / tone / worst-class calibration): misses overall, melanoma, darker tones; benign reassured; dermoscopy rate

Output: one table with a column per run and **v2 mean ± SD over seeds** — the numbers that go into the paper.
**Inputs:** attach the v2 seed notebooks (and notebook-o2 for v1, optional). CPU is fine (~20–40 min per run); GPU makes it faster.

In [ ]:
import os, time, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from scipy.optimize import minimize_scalar
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "/kaggle/working/stage8"; os.makedirs(OUT, exist_ok=True)

# ---- discover runs: every folder holding emb_D_fold0.npy ----
RUNS = {}
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "emb_D_fold0.npy" in files:
        base = os.path.basename(root)
        label = "v1_s42" if base == "stage2" else base.replace("stage2v2_", "v2_")
        RUNS.setdefault(label, root)
RUNS = dict(sorted(RUNS.items()))
assert RUNS, "attach the training notebooks (v2 seeds; optionally notebook-o2 for v1)"
folds_csv = None
for d in RUNS.values():
    for cand in [f"{d}/milk10k_lesions_folds.csv", f"{os.path.dirname(d)}/milk10k_lesions_folds.csv"]:
        if os.path.exists(cand): folds_csv = cand; break
    if folds_csv: break
assert folds_csv, "milk10k_lesions_folds.csv not found next to any run"
les = pd.read_csv(folds_csv)
print(device, "| runs:", RUNS, "\nfolds:", folds_csv)

SEED = 42                     # split seed (inner validation split used by every run) - never change
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
y_all = y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y)
MODS = ["C", "D", "M"]; PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE); y_act = is_act_cls[y]
calib_group = np.array([("MEL" if CLASSES[k] in ("MEL", "MAL_OTH") else CLASSES[k]) for k in y]); ACT_GROUPS = ["AKIEC", "BCC", "MEL", "SCCKA"]
tone = pd.cut(les["skin_tone_class"], [-1, 2, 3, 5], labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str).values
TONES = ["darker(0-2)", "medium(3)", "lighter(4-5)"]; eps = 1e-9

## A. Fusion (code identical to notebook 03)

In [ ]:
MONET = [c[:-2] for c in les.columns if c.startswith("MONET_") and c.endswith("_C")]
site = les["site"].fillna("unknown")
meta_raw = np.column_stack([les["age_approx"].fillna(-1).values, les["age_approx"].isna().values, les["skin_tone_class"].values,
                            (les["sex"] == "male").values, pd.get_dummies(site).values]).astype(np.float32)
STAGE2 = None                 # set per run in the main loop
def load_emb(mod, f): return np.load(f"{STAGE2}/emb_{mod}_fold{f}.npy").astype(np.float32)
def fold_inputs(f):
    tr_mask = les["fold"].values != f; X = {}
    for m in ["C", "D"]:
        X[m] = np.concatenate([load_emb(m, f), les[[f"{c}_{m}" for c in MONET]].values.astype(np.float32)], 1)
    X["M"] = meta_raw.copy(); age = X["M"][:, 0]; age[age < 0] = np.median(age[tr_mask & (age >= 0)])
    for m in MODS:
        mu, sd = X[m][tr_mask].mean(0), X[m][tr_mask].std(0) + 1e-6; X[m] = (X[m] - mu) / sd
    return X
def backbone_val_ids(f):
    trval = les[les["fold"] != f]
    _, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SEED)
    return les.index.isin(va.index)

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, dims, kind="xattn", d=256, drop=0.3):
        super().__init__()
        self.kind = kind
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Dropout(0.2 if m != "M" else 0.0), nn.Linear(dims[m], d),
                                                    nn.GELU(), nn.LayerNorm(d)) for m in MODS})
        if kind == "late":
            self.heads = nn.ModuleDict({m: nn.Sequential(nn.Dropout(drop), nn.Linear(d, K)) for m in MODS})
        else:
            self.mod_emb = nn.Parameter(torch.randn(3, d) * 0.02)
            self.cls = nn.Parameter(torch.zeros(1, 1, d))
            layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=0.1,
                                               batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, num_layers=2)
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, K))

    def forward(self, x, avail):                      # avail: (B,3) bool in MODS order
        tok = torch.stack([self.proj[m](x[m]) for m in MODS], 1)          # (B,3,d)
        if self.kind == "late":
            logits = torch.stack([self.heads[m](tok[:, i]) for i, m in enumerate(MODS)], 1)
            w = avail.float().unsqueeze(-1)
            return (logits * w).sum(1) / w.sum(1).clamp(min=1)
        tok = tok + self.mod_emb
        seq = torch.cat([self.cls.expand(len(tok), -1, -1), tok], 1)
        pad = torch.cat([torch.zeros(len(tok), 1, dtype=torch.bool, device=tok.device), ~avail], 1)
        return self.head(self.enc(seq, src_key_padding_mask=pad)[:, 0])

def pattern_mask(p, n):
    return torch.tensor([[m in p for m in MODS]] * n, dtype=torch.bool, device=device)

NONEMPTY = torch.tensor([[m in p for m in MODS] for p in PATTERNS], dtype=torch.bool)

def sample_avail(n, mode, fixed=None):
    """mode: 'fixed' (dedicated), 'none' (always full), 'dropout' (50% full, else a uniform non-empty subset)."""
    if mode == "fixed": return pattern_mask(fixed, n)
    full = torch.ones(n, 3, dtype=torch.bool)
    if mode == "none": return full.to(device)
    sub = NONEMPTY[torch.randint(len(PATTERNS), (n,))]
    keep_full = torch.rand(n) < 0.5
    return torch.where(keep_full.unsqueeze(1), full, sub).to(device)

In [ ]:
def evaluate(y, P):
    pred = P.argmax(1)
    out = {"bal_acc": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
           "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)])}
    try: out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

def to_t(X, idx): return {m: torch.tensor(X[m][idx], device=device) for m in MODS}

@torch.no_grad()
def predict(model, X, idx, pattern):
    model.eval(); x = to_t(X, idx)
    return model(x, pattern_mask(pattern, len(idx))).float().softmax(1).cpu().numpy()

def train_fusion(X, f, kind, mode, fixed=None, epochs=80, patience=15, bs=256, lr=3e-4):
    torch.manual_seed(SEED + f); np.random.seed(SEED + f)
    is_val = backbone_val_ids(f)
    tr_idx = np.where((les["fold"].values != f) & ~is_val)[0]
    va_idx = np.where(is_val)[0]
    val_pattern = fixed if mode == "fixed" else "CDM"
    n = np.bincount(y_all[tr_idx], minlength=K).astype(float)
    w = np.sqrt(n.max() / np.maximum(n, 1)); w = torch.tensor(w / w.mean(), dtype=torch.float, device=device)
    model = FusionNet({m: X[m].shape[1] for m in MODS}, kind).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    Xtr, ytr = to_t(X, tr_idx), torch.tensor(y_all[tr_idx], device=device)
    best, bad, state = -1, 0, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr_idx), device=device)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            xb = {m: Xtr[m][b] for m in MODS}
            loss = F.cross_entropy(model(xb, sample_avail(len(b), mode, fixed)), ytr[b],
                                   weight=w, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        # early stopping on out-of-sample lesions; with dropout, score = mean over all 7 patterns
        pats = [val_pattern] if mode != "dropout" else PATTERNS
        score = np.mean([evaluate(y_all[va_idx], predict(model, X, va_idx, p))["macro_f1"] for p in pats])
        if score > best: best, bad, state = score, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    return model, ep + 1

In [ ]:
CONFIGS = {f"dedicated_{p}": dict(kind="late", mode="fixed", fixed=p) for p in PATTERNS}
CONFIGS["late_drop"] = dict(kind="late", mode="dropout")
def run_fusion():
    rows, probs = [], {}
    for f in range(5):
        X = fold_inputs(f); te_idx = np.where(fold == f)[0]
        for name, cfg in CONFIGS.items():
            model, n_ep = train_fusion(X, f, **cfg)
            for p in ([cfg["fixed"]] if cfg["mode"] == "fixed" else PATTERNS):
                P = predict(model, X, te_idx, p)
                probs.setdefault((name, p), np.zeros((n, K), np.float32))[te_idx] = P
                rows.append(dict(config=name, fold=f, pattern=p, **evaluate(y_all[te_idx], P)))
    return pd.DataFrame(rows), probs

## B–D. Calibration, conformal sizes, dermoscopy policy, conformal triage (code from notebooks 04b, 05, 05b)

In [ ]:
def temp(P, T):
    L = np.log(P + eps) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def nll(P, yy): return -np.log(P[np.arange(len(yy)), yy] + eps).mean()
def calibrate(P):
    out = np.zeros_like(P, dtype=float)
    for f in range(5):
        tr, te = fold != f, fold == f
        T = minimize_scalar(lambda t: nll(temp(P[tr], t), y[tr]), bounds=(0.2, 5), method="bounded").x
        out[te] = temp(P[te], T)
    return out
def q_upper(v, a):
    v = np.sort(v); m = len(v); k = int(np.ceil((m + 1) * (1 - a)))
    return np.inf if (m == 0 or k > m) else v[k - 1]
def set_size(P, alpha=0.10):
    S = np.zeros((n, K), bool); st = 1 - P[np.arange(n), y]
    for f in range(5):
        cal, te = fold != f, fold == f; S[te] = (1 - P[te]) <= q_upper(st[cal], alpha)
    return S.sum(1).mean(), S[np.arange(n), y].mean()

# ---- policy ----
TARGET_SPEC = 0.70; RATES = np.round(np.arange(0, 1.0001, 0.05), 2)
def act_score(P): return P[:, is_act_cls].sum(1)
def spec_thresholds(P):
    t = np.zeros(n)
    for f in range(5): t[fold == f] = np.quantile(act_score(P)[(fold != f) & ~y_act], TARGET_SPEC)
    return t
def policy_metrics(P_CM, P_CDM, thr_CM, thr_CDM, req):
    P = np.where(req[:, None], P_CDM, P_CM); flag = act_score(P) >= np.where(req, thr_CDM, thr_CM)
    return {"macro_f1": f1_score(y, P.argmax(1), average="macro", labels=range(K), zero_division=0),
            "act_sens@spec": (flag & y_act).sum() / y_act.sum()}
def run_policy(P_CM, P_CDM):
    tCM, tCDM = spec_thresholds(P_CM), spec_thresholds(P_CDM); top2 = np.sort(P_CM, 1)[:, -2:]
    scores = {"random": np.random.RandomState(0).rand(n), "least_conf": 1 - P_CM.max(1),
              "entropy": -(P_CM * np.log(P_CM + eps)).sum(1), "small_margin": -(top2[:, 1] - top2[:, 0])}
    rows = []
    for name, s in scores.items():
        order = np.argsort(-s)
        for q in RATES:
            req = np.zeros(n, bool); req[order[:int(round(q * n))]] = True
            rows.append(dict(policy=name, rate=q, **policy_metrics(P_CM, P_CDM, tCM, tCDM, req)))
    cur = pd.DataFrame(rows); out = {}
    for m in ["macro_f1", "act_sens@spec"]:
        lo, hi = cur.loc[cur.rate == 0, m].iloc[0], cur.loc[cur.rate == 1, m].iloc[0]; tgt = lo + 0.95 * (hi - lo)
        for name, d in cur.groupby("policy"):
            ok = d[d[m] >= tgt]; out[f"rate95_{m}_{name}"] = ok["rate"].min() if len(ok) else np.nan
        out[f"{m}_CM_only"], out[f"{m}_CDM_all"] = lo, hi
        out[f"{m}_least_conf@30%"] = cur[(cur.policy == "least_conf") & (cur.rate == 0.30)][m].iloc[0]
    return out, cur

# ---- conformal triage (two-stage, binary) ----
def tri_sets(P, cal_fn, aA, aB, variant):
    s = act_score(P); inA, inB = np.zeros(n, bool), np.zeros(n, bool)
    for f in range(5):
        te, cal = fold == f, cal_fn(f)
        inB[te] = s[te] <= q_upper(s[cal & ~y_act], aB)
        if variant == "group":
            inA[te] = (1 - s[te]) <= q_upper(1 - s[cal & y_act], aA)
        elif variant == "worst_class":
            inA[te] = (1 - s[te]) <= max(q_upper(1 - s[cal & (calib_group == g)], aA) for g in ACT_GROUPS)
        elif variant == "tone":
            for g in TONES:
                m = te & (tone == g); inA[m] = (1 - s[m]) <= q_upper(1 - s[cal & y_act & (tone == g)], aA)
    return inA, inB
def decide(inA, inB): return np.where(inA & ~inB, "refer", np.where(inB & ~inA, "reassure", "ambiguous"))
def triage(P_CM, P_CDM, alpha, aB, strategy, variant):
    if strategy == "all":
        d = decide(*tri_sets(P_CDM, lambda f: fold != f, alpha, aB, variant)); return np.where(d == "ambiguous", "refer", d), np.ones(n, bool)
    aA = alpha / 2 if strategy == "adaptive" else alpha
    d1 = decide(*tri_sets(P_CM, lambda f: fold != f, aA, aB, variant))
    if strategy == "none": return np.where(d1 == "ambiguous", "refer", d1), np.zeros(n, bool)
    req = d1 == "ambiguous"; d2 = decide(*tri_sets(P_CDM, lambda f: (fold != f) & req, aA, aB, variant))
    final = d1.copy(); final[req] = np.where(d2[req] == "ambiguous", "refer", d2[req]); return final, req
def tri_summary(final, req):
    re = final == "reassure"; dk = tone == "darker(0-2)"; mel = calib_group == "MEL"
    return {"dermoscopy_rate": req.mean(), "act_missed": (re & y_act).sum() / y_act.sum(),
            "MEL_missed": (re & mel).sum() / mel.sum(), "darker_missed": (re & y_act & dk).sum() / (y_act & dk).sum(),
            "benign_reassured": (re & ~y_act).sum() / (~y_act).sum(), "referral_rate": (final == "refer").mean()}

## Main loop (resumable: each run's results are saved as soon as it finishes)

In [ ]:
ALL = {}
for label, d in RUNS.items():
    STAGE2 = d; f_out = f"{OUT}/{label}"; os.makedirs(f_out, exist_ok=True); t0 = time.time()
    if os.path.exists(f"{f_out}/key_metrics.csv"):
        ALL[label] = pd.read_csv(f"{f_out}/key_metrics.csv", index_col=0).iloc[:, 0]; print("loaded", label); continue
    km = {}
    for m in ["C", "D"]:                                   # unimodal CNNs
        o = pd.read_csv(f"{d}/oof_{m}.csv").set_index("lesion_id").reindex(les["lesion_id"])
        e = evaluate(y, o[CLASSES].values); km[f"unimodal_{m}_macro_f1"], km[f"unimodal_{m}_auc"] = e["macro_f1"], e["macro_auc"]
        km[f"unimodal_{m}_MEL_recall"] = recall_score(y == 7, o[CLASSES].values.argmax(1) == 7)
    res, probs = run_fusion(); res.to_csv(f"{f_out}/fusion_long.csv", index=False)
    np.savez_compressed(f"{f_out}/oof_probs_fusion.npz", **{f"{c}|{p}": v for (c, p), v in probs.items()})
    mean = res.groupby(["config", "pattern"])[["macro_f1", "macro_auc"]].mean()
    for p in PATTERNS:
        km[f"late_drop_{p}_macro_f1"] = mean.loc[("late_drop", p), "macro_f1"]; km[f"late_drop_{p}_auc"] = mean.loc[("late_drop", p), "macro_auc"]
        km[f"gap_vs_dedicated_{p}"] = mean.loc[("late_drop", p), "macro_f1"] - mean.loc[(f"dedicated_{p}", p), "macro_f1"]
    CAL = {p: calibrate(probs[("late_drop", p)].astype(float)) for p in PATTERNS}
    for p in PATTERNS: km[f"set_size90_{p}"], km[f"coverage90_{p}"] = set_size(CAL[p])
    pol, cur = run_policy(CAL["CM"], CAL["CDM"]); km.update(pol); cur.to_csv(f"{f_out}/policy_curves.csv", index=False)
    for variant in ["group", "tone", "worst_class"]:
        for strat in ["none", "adaptive", "all"]:
            for k, v in tri_summary(*triage(CAL["CM"], CAL["CDM"], 0.10, 0.20, strat, variant)).items():
                km[f"triage_{variant}_{strat}_{k}"] = v
    ALL[label] = pd.Series(km); ALL[label].to_csv(f"{f_out}/key_metrics.csv")
    print(f"{label} done in {(time.time() - t0) / 60:.1f} min")

## Paper table: every run + v2 mean ± SD over seeds

In [ ]:
tab = pd.DataFrame(ALL)
v2 = [c for c in tab.columns if c.startswith("v2_")]
if len(v2) >= 2:
    tab["v2 mean ± SD"] = tab[v2].mean(1).round(3).astype(str) + " ± " + tab[v2].std(1).round(3).astype(str)
tab.to_csv(f"{OUT}/paper_table_all_runs.csv")
SHOW = (["unimodal_C_macro_f1", "unimodal_D_macro_f1", "unimodal_C_auc", "unimodal_D_auc", "unimodal_D_MEL_recall",
         "late_drop_CDM_macro_f1", "late_drop_CM_macro_f1", "late_drop_CDM_auc", "late_drop_CM_auc"]
        + [f"gap_vs_dedicated_{p}" for p in PATTERNS] + [f"set_size90_{p}" for p in ["C", "CM", "D", "DM", "CDM"]]
        + ["rate95_macro_f1_least_conf", "rate95_macro_f1_random", "rate95_act_sens@spec_least_conf", "macro_f1_least_conf@30%",
           "act_sens@spec_CM_only", "act_sens@spec_CDM_all"]
        + [f"triage_{v}_adaptive_{k}" for v in ["group", "tone", "worst_class"]
           for k in ["dermoscopy_rate", "act_missed", "MEL_missed", "darker_missed", "benign_reassured"]])
pd.set_option("display.max_rows", 200)
tab.loc[SHOW].apply(lambda col: col.map(lambda v: round(v, 3) if isinstance(v, (float, np.floating)) else v))